In [ ]:
# csed

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.stats import chi2_contingency

import os
import re

from scipy.stats import gaussian_kde, ks_2samp, mannwhitneyu

In [20]:
DIR = "./data/raw/"
files = os.listdir(DIR)
files = sorted(files)

for x in files:
    print(x)

01-Grades_CSE100_2022-4_Fall.xlsx
02-Grades_CSE100_2023-1_Winter.xlsx
03-Grades_CSE100_2023-4_Fall.xlsx
04-Grades_CSE100_2024-1_Winter.xlsx
05-Grades_CSE100_2024-2_Spring.xlsx
06-Grades_CSE100_2024-4_Fall.xlsx
07-Grades_CSE100_2025-1_Winter.xlsx
08-Grades_CSE100_2025-4_Fall.xlsx


## Read and preprocess files from Winter 2025 and Fall 2025

In [28]:
def get_df(f):
    """ Read in the grades excel file, skip first row """
    print(f'Reading {f}')
    df = pd.read_excel(DIR + f)  # Skip the first two rows of metadata
    df = df.iloc[1:] # Remove the first row ("points out of")
    return df

df1 = get_df(files[6])
df2 = get_df(files[7])

df1['quarter'] = 'winter'
df1['year'] = 2025
df1['course'] = df1['quarter'].astype(str) + df1['year'].astype(str)

df2['quarter'] = 'fall'
df2['year'] = 2025
df2['course'] = df2['quarter'].astype(str) + df2['year'].astype(str)

def select_and_rename(df):
    """ Selects columns from df that have '.1' in their name, which are 
        duplicate column names. The duplicate columns are the ones we care 
        about, and the original columns have raw point data.
        Renames them by removing the ' (1)' suffix, 
        and returns the resulting dataframe.
    """
    renamed = {}
    df_cols = []
    new_df_cols = []
    for x in df.columns:
        if '.1' in x:
            renamed[x] = x.split(' (')[0]
            df_cols.append(x)
            new_df_cols.append(renamed[x])
        if x in ['quarter', 'course',
                 'Preparation', 'Application', 'Examination', 
                 'Total']:
            df_cols.append(x)
            new_df_cols.append(x)

    df = df[df_cols]
    df = df.rename(columns=renamed)
    return df


df1_sel = select_and_rename(df1)
df2_sel = select_and_rename(df2)

df1_sel['min_category'] = df1_sel[['Preparation', 'Application', 'Examination']].idxmin(axis=1)
df2_sel['min_category'] = df2_sel[['Preparation', 'Application', 'Examination']].idxmin(axis=1)

df1_sel.rename(columns={'Total': 'Overall'}, inplace=True)
df2_sel.rename(columns={'Total': 'Overall'}, inplace=True)

df1_sel['atrisk'] = df1_sel['Overall'] < 57.5
df2_sel['atrisk'] = df2_sel['Overall'] < 57.5

df = pd.concat([df1_sel, df2_sel], ignore_index=True)
df_sub = df[['Preparation','Application', 'Examination', 'Overall', 'course', 'min_category', 'atrisk']]

print(f"Description of df_sub:\n{df_sub.describe()}\n")
print(f"Description of df1_sel:\n{df1_sel.describe()}\n")
print(f"Description of df2_sel:\n{df2_sel.describe()}\n")



Reading 07-Grades_CSE100_2025-1_Winter.xlsx
Reading 08-Grades_CSE100_2025-4_Fall.xlsx
Description of df_sub:
       Preparation  Application  Examination     Overall
count   852.000000    852.00000   852.000000  852.000000
mean     91.192043     94.74877    90.772863   86.740027
std      10.968626      8.77115    11.250158   12.910712
min      36.260000     23.66700     0.000000    0.000000
25%      88.114583     93.50000    88.656771   82.160833
50%      95.506410     98.00000    94.366023   91.282292
75%      99.666667    100.00000    97.500000   95.674419
max     100.000000    100.00000   100.000000  100.000000

Description of df1_sel:
          Midterm       Final   Project 1   Project 2  \
count  491.000000  491.000000  491.000000  491.000000   
mean     0.935628    0.904992    0.876558    0.910183   
std      0.127028    0.129105    0.230908    0.198484   
min      0.000000    0.000000    0.000000    0.000000   
25%      0.929412    0.879767    0.900000    0.850000   
50%      0.

## Chi-squared tests for distribution of students' min grade categories

In [31]:
def get_ct_from_df(ct_df, feature='atrisk'):
    ct = pd.crosstab(ct_df[feature], ct_df["min_category"])
    ct = ct[['Preparation', 'Application', 'Examination']]
    
    chi2, p, dof, expected = chi2_contingency(ct)

    return ct, chi2, p, dof, expected

def fmt_p(p):
    if p < 0.001:
        return f"{p:.2e}".replace("e-16", r"\times 10^{-16}")  # optional
    return f"{p:.2f}"

def get_props(ct):
    props = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)
    print(props, '\n')
    return props

def latex_min_category_table(df_sub):
    
    # Overall Q1 vs Q2
    ct_all, chi2_all, p_all, dof_all, expected_all = get_ct_from_df(
        df_sub,
        feature='course')
    props_all = get_props(ct_all)

    # Pass/fail within Q1
    ct_q1, chi2_q1, p_q1, dof_q1, expected_q1 = get_ct_from_df(
        df_sub[df_sub["course"] == "winter2025"],
        feature='atrisk'
    )
    props_q1 = get_props(ct_q1)

    # Pass/fail within Q2
    ct_q2, chi2_q2, p_q2, dof_q2, expected_q2 = get_ct_from_df(
        df_sub[df_sub["course"] == "fall2025"],
        feature='atrisk'
    )
    props_q2 = get_props(ct_q2)

    latex = rf"""
\begin{{table}}[t]
\centering
\caption{{Distribution of students' lowest grade categories.}}
\label{{tab:min-category-distribution}}
\begin{{tabular}}{{lcccc}}
\toprule
Group & Prep. & Appl. & Exam. & $\chi^2$ $p$-value \\
\midrule
Q1 (remote) &
{props_all.loc['winter2025', 'Preparation']}\% &
{props_all.loc['winter2025', 'Application']}\% &
{props_all.loc['winter2025', 'Examination']}\% &
\multirow{{2}}{{*}}{{{p_all:.2e}}} \\
Q2 (in-person) &
{props_all.loc['fall2025', 'Preparation']}\% &
{props_all.loc['fall2025', 'Application']}\% &
{props_all.loc['fall2025', 'Examination']}\% &
\\
\midrule
Passing Q1 &
{props_q1.loc[False, 'Preparation']}\% &
{props_q1.loc[False, 'Application']}\% &
{props_q1.loc[False, 'Examination']}\% &
\multirow{{2}}{{*}}{{{p_q1:.2f}}} \\
Failing Q1 &
{props_q1.loc[True, 'Preparation']}\% &
{props_q1.loc[True, 'Application']}\% &
{props_q1.loc[True, 'Examination']}\% &
\\
\midrule
Passing Q2 &
{props_q2.loc[False, 'Preparation']}\% &
{props_q2.loc[False, 'Application']}\% &
{props_q2.loc[False, 'Examination']}\% &
\multirow{{2}}{{*}}{{{p_q2:.2f}}} \\
Failing Q2 &
{props_q2.loc[True, 'Preparation']}\% &
{props_q2.loc[True, 'Application']}\% &
{props_q2.loc[True, 'Examination']}\% &
\\
\bottomrule
\end{{tabular}}
\end{{table}}
    """
    print(latex)
    
latex_min_category_table(df_sub)

min_category  Preparation  Application  Examination
course                                             
fall2025             30.5          8.0         61.5
winter2025           48.7         18.9         32.4 

min_category  Preparation  Application  Examination
atrisk                                             
False                48.9         18.7         32.3
True                 42.9         23.8         33.3 

min_category  Preparation  Application  Examination
atrisk                                             
False                30.9          8.4         60.7
True                 20.0          0.0         80.0 


\begin{table}[t]
\centering
\caption{Distribution of students' lowest grade categories.}
\label{tab:min-category-distribution}
\begin{tabular}{lcccc}
\toprule
Group & Prep. & Appl. & Exam. & $\chi^2$ $p$-value \\
\midrule
Q1 (remote) &
48.7\% &
18.9\% &
32.4\% &
\multirow{2}{*}{1.07e-16} \\
Q2 (in-person) &
30.5\% &
8.0\% &
61.5\% &
\\
\midrule
Passing Q1 &
48.9\% &
